# CEG5305 Part 2, Lecture 2, session 01: the DPO practice homework (no credit), in Colab

This notebook runs [`dpo_base.py`](https://costa-nus.github.io/CEG5305_GenAI/part2/lecture-2/code/dpo_base.py): the chat model Qwen2.5-0.5B-Instruct (the SFT model, $\pi_{\mathrm{ref}}$) learns a preference from pairs (prompt, chosen, rejected) with the DPO loss of Part B. Only a LoRA update on the query and value projections is trained, as in Lecture 1, session 02. Two tasks: `format` (reply in a fixed QA format; it is chat text, not code) and `restaurant` (your own preference over restaurants). `zero` is the instruction alone; `dpo` trains, with three seeds.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Run time per seed on a T4: TO BE MEASURED ON COLAB.

In [ ]:
# 1. A GPU must be on: Runtime -> Change runtime type -> T4 GPU
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run this cell again"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Get the base code. If the address below is not published yet, upload dpo_base.py by hand
#    (file panel on the left, Upload) before running this cell: a copy that is already here is kept.
import os
if not os.path.exists("dpo_base.py"):
    !wget -q -O dpo_base.py https://costa-nus.github.io/CEG5305_GenAI/part2/lecture-2/code/dpo_base.py || rm -f dpo_base.py
assert os.path.exists("dpo_base.py"), "dpo_base.py is not here: upload it in the file panel on the left"
!python dpo_base.py --help | head -40

In [ ]:
# 3. Check the setup: every line must say PASS. The first run downloads the model (about 1 GB).
#    A warning about HF_TOKEN can be ignored: no account is needed.
!python dpo_base.py --selfcheck

In [ ]:
# 4. Task 1, the QA format. zero: the instruction only, no training (one run: no randomness)
!python dpo_base.py --task format --method zero --out format_zero.json

In [ ]:
# 5. Task 1, the QA format. dpo: train LoRA with the DPO loss, three seeds
!python dpo_base.py --task format --method dpo --out format_dpo.json

In [ ]:
# 6. Task 2, the example restaurant preference (quiet, vegetarian, under S$20). zero, then dpo
!python dpo_base.py --task restaurant --method zero --out restaurant_zero.json
!python dpo_base.py --task restaurant --method dpo --out restaurant_dpo.json

## 7. Your data

Write your own instruction and pairs for one task. One pair per line: `prompt`, `chosen`, `rejected`, and for the restaurant score `want` and `avoid` (a word that only the right place has, and one that only the wrong place has). The last 25% of the lines are the test pairs, or put the test pairs in a second file and pass `--test`. Use new places in the test pairs. Write your predictions down **before** you run: how often does the model answer as you prefer with the instruction alone, and after DPO?

In [ ]:
# The two lines below are placeholders: replace them with your own pairs (8 or more), then run the cell.
import json
my_pairs = [
    {"prompt": "Which place do you recommend for dinner tonight?\nA: ... \nB: ...",
     "chosen": "I recommend ...", "rejected": "I recommend ...", "want": "...", "avoid": "..."},
    {"prompt": "...", "chosen": "...", "rejected": "...", "want": "...", "avoid": "..."},
]
with open("my_prefs.jsonl", "w") as f:
    for p in my_pairs:
        f.write(json.dumps(p) + "\n")
with open("my_instruction.txt", "w") as f:      # your instruction (the system message); a file, so quotes are safe
    f.write("You are a restaurant guide. Recommend one of the places in one sentence.")
!python dpo_base.py --selfcheck --task restaurant --data my_prefs.jsonl --instruction @my_instruction.txt

In [ ]:
# 8. Your runs: change ONE thing at a time (the instruction, the pairs, --beta, --sft, --rank, or one value
#    in FIXED), keep the three seeds, give every run its own --out file. For the format task use --task format.
!python dpo_base.py --task restaurant --method zero --data my_prefs.jsonl --instruction @my_instruction.txt --out my_zero.json
!python dpo_base.py --task restaurant --method dpo  --data my_prefs.jsonl --instruction @my_instruction.txt --out my_dpo.json

In [ ]:
# 9. The results table: one line per JSON file in this folder, then the training curves of the dpo runs
import glob, json
import matplotlib.pyplot as plt

pct = lambda v: "   -" if v is None else f"{100 * v:3.0f}%"
print(f"{'file':28s} {'task':10s} {'method':6s} {'beta':>5s} {'sft':>4s}  {'pref acc':>9s} {'score':>6s}  "
      f"{'after dpo: pref acc':>20s} {'reward acc':>11s} {'score (per seed)':>24s}")
curves = {}
for path in sorted(glob.glob("*.json")):
    d = json.load(open(path))
    if "results" not in d or "settings" not in d:
        continue
    s, b, runs = d["settings"], d["results"]["before"], d["results"]["runs"]
    after = d["results"].get("mean_after", {})
    per_seed = "/".join(pct(r["after"]["score_pass"]).strip() for r in runs)
    print(f"{path[:-5]:28s} {s['task']:10s} {s['method']:6s} {str(s['beta'] or ''):>5s} {str(s.get('sft') or ''):>4s}  "
          f"{pct(b['pref_acc']):>9s} {pct(b['score_pass']):>6s}  {pct(after.get('pref_acc')):>20s} "
          f"{pct(after.get('reward_acc')):>11s} {per_seed:>24s}")
    if d["curves"]:
        curves[path[:-5]] = list(d["curves"].values())

if curves:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for name, per_seed in curves.items():
        steps = per_seed[0]["step"]
        for ax, key in zip(axes, ("loss", "margin", "chosen_nll")):
            ax.plot(steps, [sum(c[key][i] for c in per_seed) / len(per_seed) for i in range(len(steps))],
                    label=f"{name} (mean of {len(per_seed)} seeds)")
    for ax, label in zip(axes, ("DPO loss", "reward margin, beta * log ratio", "chosen NLL per token")):
        ax.set_xlabel("training step"); ax.set_ylabel(label); ax.grid(alpha=.3)
    axes[0].legend()
    plt.show()

In [ ]:
# 10. Read the replies, not only the numbers: the test replies before training and after, for the first seed
import json
d = json.load(open("format_dpo.json"))        # change the file name
for before, after in zip(d["results"]["before"]["replies"], d["results"]["runs"][0]["after"]["replies"]):
    print("PROMPT:", before["prompt"].replace("\n", " | "))
    print("  before:", before["score"], repr(before["reply"]))
    print("  after: ", after["score"], repr(after["reply"]))